# EEG Seizure Detection Training Notebook

This notebook fixes the `cross_val_predict only works for partitions` error by using:

- `StratifiedKFold` for out-of-fold predictions used in ROC, PR, confusion matrix
- a separate repeated CV loop with `RepeatedStratifiedKFold` for more stable model comparison scores

It is designed for the common **Epileptic Seizure Recognition** CSV format with a target column named `y`.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 3
CSV_PATH = 'Epileptic Seizure Recognition.csv'  # change this if needed
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


## 1) Load and prepare data

This maps seizure class `1` to positive and all other classes to `0` for binary classification.


In [ ]:
def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    for col in ['Unnamed', 'Unnamed: 0']:
        if col in df.columns:
            df = df.drop(columns=[col])

    if 'y' not in df.columns:
        raise ValueError("Expected target column 'y' in dataset.")

    y = df['y'].apply(lambda v: 1 if int(v) == 1 else 0).astype(int)
    X = df.drop(columns=['y'])
    X = X.select_dtypes(include=[np.number]).copy()
    return X, y

X, y = load_dataset(CSV_PATH)
print('Loaded dataset shape:', X.shape)
print('Positive rate:', round(y.mean(), 4))
display(pd.Series(y).value_counts().rename(index={0: 'non-seizure', 1: 'seizure'}).to_frame('count'))


In [ ]:
plt.figure(figsize=(6,4))
counts = y.value_counts().sort_index()
plt.bar(['non-seizure', 'seizure'], counts.values)
plt.title('Class Balance')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 2) Preprocessing and model definitions


In [ ]:
def build_preprocessor(X):
    numeric_features = X.columns.tolist()
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    return ColumnTransformer(
        [('num', numeric_transformer, numeric_features)],
        remainder='drop',
        verbose_feature_names_out=False,
    )

preprocessor = build_preprocessor(X)

def candidate_models(preprocessor):
    return {
        'logreg': Pipeline([
            ('preprocess', preprocessor),
            ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE))
        ]),
        'random_forest': Pipeline([
            ('preprocess', preprocessor),
            ('model', RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                n_jobs=-1,
                class_weight='balanced_subsample',
                random_state=RANDOM_STATE,
            ))
        ]),
        'extra_trees': Pipeline([
            ('preprocess', preprocessor),
            ('model', ExtraTreesClassifier(
                n_estimators=700,
                min_samples_leaf=2,
                n_jobs=-1,
                class_weight='balanced',
                random_state=RANDOM_STATE,
            ))
        ]),
        'hist_gb': Pipeline([
            ('preprocess', preprocessor),
            ('model', HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_depth=8,
                max_iter=300,
                random_state=RANDOM_STATE,
            ))
        ]),
    }

models = candidate_models(preprocessor)
list(models.keys())


## 3) Stable repeated-CV model comparison

This section does **not** use `cross_val_predict`. It manually aggregates fold scores across repeated CV.


In [ ]:
def evaluate_repeated_cv(name, model, X, y):
    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    rows = []
    X_df = X.reset_index(drop=True)
    y_sr = y.reset_index(drop=True)

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_df, y_sr), start=1):
        model_fold = clone(model)
        X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
        y_train, y_test = y_sr.iloc[train_idx], y_sr.iloc[test_idx]

        model_fold.fit(X_train, y_train)
        y_pred = model_fold.predict(X_test)
        y_prob = model_fold.predict_proba(X_test)[:, 1]

        rows.append({
            'fold': fold_idx,
            'accuracy': accuracy_score(y_test, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_prob),
        })

    scores = pd.DataFrame(rows)
    summary = scores.mean(numeric_only=True).to_dict()
    summary['model'] = name
    return scores, summary

all_fold_scores = {}
leaderboard_rows = []

for name, model in models.items():
    fold_scores, summary = evaluate_repeated_cv(name, model, X, y)
    all_fold_scores[name] = fold_scores
    leaderboard_rows.append(summary)

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    by=['roc_auc', 'f1', 'accuracy'], ascending=False
).reset_index(drop=True)
leaderboard


In [ ]:
leaderboard.to_csv(OUTPUT_DIR / 'leaderboard.csv', index=False)

plot_df = leaderboard.set_index('model')[['accuracy', 'f1', 'roc_auc']]
x = np.arange(len(plot_df.index))
width = 0.25

plt.figure(figsize=(8,5))
plt.bar(x - width, plot_df['accuracy'], width=width, label='accuracy')
plt.bar(x, plot_df['f1'], width=width, label='f1')
plt.bar(x + width, plot_df['roc_auc'], width=width, label='roc_auc')
plt.xticks(x, plot_df.index, rotation=20)
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Model Comparison')
plt.legend()
plt.tight_layout()
plt.show()


## 4) Out-of-fold predictions for plots

This is where `cross_val_predict` is safe, because we use plain `StratifiedKFold`, which forms a proper partition.


In [ ]:
best_name = leaderboard.iloc[0]['model']
best_model = models[best_name]
oof_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

y_pred_oof = cross_val_predict(best_model, X, y, cv=oof_cv, method='predict', n_jobs=-1)
y_prob_oof = cross_val_predict(best_model, X, y, cv=oof_cv, method='predict_proba', n_jobs=-1)[:, 1]

print('Best model:', best_name)
print(classification_report(y, y_pred_oof, digits=4))


In [ ]:
cm = confusion_matrix(y, y_pred_oof)
plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion Matrix')
plt.xticks([0, 1], ['non-seizure', 'seizure'])
plt.yticks([0, 1], ['non-seizure', 'seizure'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y, y_prob_oof)
roc_auc = roc_auc_score(y, y_prob_oof)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

precision, recall, _ = precision_recall_curve(y, y_prob_oof)
plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.tight_layout()
plt.show()


## 5) Fit a stronger final ensemble on the full dataset


In [ ]:
final_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', VotingClassifier(
        estimators=[
            ('et', ExtraTreesClassifier(
                n_estimators=600,
                min_samples_leaf=2,
                n_jobs=-1,
                class_weight='balanced',
                random_state=RANDOM_STATE,
            )),
            ('rf', RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=2,
                n_jobs=-1,
                class_weight='balanced_subsample',
                random_state=RANDOM_STATE,
            )),
            ('lr', LogisticRegression(
                max_iter=3000,
                class_weight='balanced',
                random_state=RANDOM_STATE,
            )),
        ],
        voting='soft',
        n_jobs=-1,
    ))
])

final_model.fit(X, y)
joblib.dump(final_model, OUTPUT_DIR / 'seizure_model.joblib')
print('Saved model to', OUTPUT_DIR / 'seizure_model.joblib')


## 6) Feature importance visualization

Permutation importance is model-agnostic and works well for the final pipeline.


In [ ]:
perm = permutation_importance(final_model, X, y, n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(20)

plt.figure(figsize=(8,7))
plt.barh(importances.index[::-1], importances.values[::-1])
plt.xlabel('Mean Permutation Importance')
plt.title('Top 20 Features')
plt.tight_layout()
plt.show()


## 7) Notes

- Your earlier error happened because `cross_val_predict` requires each sample to appear in exactly one test fold.
- `RepeatedStratifiedKFold` violates that requirement because samples appear in multiple test folds across repeats.
- So the split logic here is intentionally separated into:
  - repeated CV for score stability
  - single partitioned CV for OOF predictions and plots
